# 05 — Events Cleaning (`events_clean.csv`)

## Purpose
Clean the Yahoo Finance **earnings** announcements and the SEC **8-K** filings and stack them into the `events` table from the data model.

## Grain
One row per company event. `event_type` is `earnings` or `8-K`. Columns that do not apply to a type stay empty.

## Inputs (`data/raw/`)
- `earnings_events.csv` — Yahoo Finance earnings announcements
- `events_8k.csv` — SEC EDGAR 8-K filings

## Output
`data/processed/events_clean.csv`

| Column | Applies to | Description |
|---|---|---|
| `event_id` | all | Surrogate key (1..n, sorted by date, ticker, type) |
| `ticker` | all | Project ticker |
| `event_type` | all | `earnings` or `8-K` |
| `event_date` | all | Announcement date (earnings) or SEC filing date (8-K) |
| `reaction_date` | all | First trading day the market could react (see rules below) |
| `after_close` | earnings | True if announced after the 4 PM New York close |
| `eps_estimate`, `eps_actual`, `surprise_pct` | earnings | EPS figures from Yahoo |
| `items`, `items_description` | 8-K | Item codes (e.g. `2.02,9.01`) and their descriptions |
| `accession_number` | 8-K | SEC accession number, the unique ID of the filing (extra column for traceability) |

## Reaction-date rules
- **Earnings:** announced before the close -> same day if it is a trading day; after the close -> next trading day.
- **8-K:** the file has no filing time, so the filing date is used (next trading day if it is not a trading day). **Exception:** an 8-K containing item `2.02` (earnings release) filed on the same day as a known earnings event takes that event's reaction date, so it follows the same after-close logic.
- Trading days follow the NYSE calendar (weekends and NYSE holidays are skipped).

## Things to know
- Every earnings announcement also has a matching 8-K with item `2.02`, so both rows exist for the same news. Filter `items` containing `2.02` (or use `event_type = 'earnings'` only) when counting earnings events, to avoid double counting.
- EPS values from Yahoo are split-adjusted non-GAAP figures. They are **not** comparable with `fundamentals.eps_diluted` (as filed, GAAP).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")

EARNINGS_PATH = RAW_DIR / "earnings_events.csv"
K8_PATH = RAW_DIR / "events_8k.csv"
MARKET_PATH = PROCESSED_DIR / "market_clean.csv"      # optional: only used to cross-check the trading calendar
COMPANIES_PATH = PROCESSED_DIR / "companies.csv"      # optional: used for the foreign-key check
OUTPUT_PATH = PROCESSED_DIR / "events_clean.csv"

TICKERS = ["AAPL", "JPM", "MSFT", "NVDA", "JNJ", "XOM", "KO", "AMZN", "LLY", "TSLA"]
PROJECT_TICKERS = set(TICKERS)
START, END = pd.Timestamp("2021-01-01"), pd.Timestamp("2025-12-31")   # project analysis period

## 1. NYSE trading calendar

Weekends plus the NYSE full-day holidays (including the 2025-01-09 market closure for the National Day of Mourning). If `market_clean.csv` exists, the calendar is cross-checked against the real price dates.

In [ ]:
NYSE_HOLIDAYS = pd.to_datetime([
    # 2021
    "2021-01-01", "2021-01-18", "2021-02-15", "2021-04-02", "2021-05-31", "2021-07-05", "2021-09-06", "2021-11-25", "2021-12-24",
    # 2022
    "2022-01-17", "2022-02-21", "2022-04-15", "2022-05-30", "2022-06-20", "2022-07-04", "2022-09-05", "2022-11-24", "2022-12-26",
    # 2023
    "2023-01-02", "2023-01-16", "2023-02-20", "2023-04-07", "2023-05-29", "2023-06-19", "2023-07-04", "2023-09-04", "2023-11-23", "2023-12-25",
    # 2024
    "2024-01-01", "2024-01-15", "2024-02-19", "2024-03-29", "2024-05-27", "2024-06-19", "2024-07-04", "2024-09-02", "2024-11-28", "2024-12-25",
    # 2025
    "2025-01-01", "2025-01-09", "2025-01-20", "2025-02-17", "2025-04-18", "2025-05-26", "2025-06-19", "2025-07-04", "2025-09-01", "2025-11-27", "2025-12-25",
    # 2026
    "2026-01-01", "2026-01-19", "2026-02-16", "2026-04-03", "2026-05-25", "2026-06-19", "2026-07-03", "2026-09-07", "2026-11-26", "2026-12-25",
])
HOLIDAY_SET = set(NYSE_HOLIDAYS)


def is_trading_day(day):
    day = pd.Timestamp(day).normalize()
    return day.dayofweek < 5 and day not in HOLIDAY_SET


def next_trading_day(day, strictly_after=False):
    """First trading day on/after `day` (or strictly after it)."""
    day = pd.Timestamp(day).normalize()
    if strictly_after:
        day += pd.Timedelta(days=1)
    while not is_trading_day(day):
        day += pd.Timedelta(days=1)
    return day


def reaction_date(event_date, after_close):
    return next_trading_day(event_date, strictly_after=bool(after_close))


# Calendar self-tests
assert reaction_date("2021-01-29", True) == pd.Timestamp("2021-02-01")      # Friday after close -> Monday
assert reaction_date("2022-04-14", True) == pd.Timestamp("2022-04-18")      # Good Friday is closed
assert reaction_date("2025-01-08", True) == pd.Timestamp("2025-01-10")      # 2025-01-09 market closure
assert reaction_date("2021-03-06", False) == pd.Timestamp("2021-03-08")     # Saturday -> Monday
assert reaction_date("2021-03-01", False) == pd.Timestamp("2021-03-01")     # trading day, before close

if MARKET_PATH.exists():
    price_days = pd.DatetimeIndex(pd.to_datetime(pd.read_csv(MARKET_PATH, usecols=["date"])["date"].drop_duplicates()))
    span = pd.date_range(price_days.min(), price_days.max(), freq="D")
    calendar_days = pd.DatetimeIndex([d for d in span if is_trading_day(d)])
    only_calendar = calendar_days.difference(price_days)
    only_prices = price_days.difference(calendar_days)
    print(f"Calendar vs market_clean.csv: {len(only_calendar)} days only in calendar, {len(only_prices)} days only in prices")
    if len(only_calendar) or len(only_prices):
        print("  only in calendar:", [str(d.date()) for d in only_calendar][:15])
        print("  only in prices:  ", [str(d.date()) for d in only_prices][:15])
else:
    print("market_clean.csv not found: calendar not cross-checked.")

## 2. Earnings announcements

Checks and cleaning: required columns, uppercase tickers, parsed dates, duplicate `(ticker, announce_date)` pairs, project tickers and the 2021-2025 period.

In [ ]:
earn = pd.read_csv(EARNINGS_PATH)
print(f"Loaded {len(earn):,} earnings rows")

required = {"ticker", "announce_datetime", "announce_date", "after_close", "reaction_date",
            "eps_estimate", "eps_actual", "surprise_pct"}
missing = required - set(earn.columns)
if missing:
    raise ValueError(f"earnings_events.csv is missing columns: {sorted(missing)}")

earn["ticker"] = earn["ticker"].astype(str).str.strip().str.upper()
earn["announce_datetime"] = pd.to_datetime(earn["announce_datetime"], errors="coerce")
earn["announce_date"] = pd.to_datetime(earn["announce_date"], errors="coerce").dt.normalize()
earn["after_close"] = earn["after_close"].astype(str).str.strip().str.lower().map({"true": True, "false": False})
for col in ["eps_estimate", "eps_actual", "surprise_pct"]:
    earn[col] = pd.to_numeric(earn[col], errors="coerce")

assert earn["announce_date"].notna().all(), "unparseable announce_date"
assert earn["after_close"].notna().all(), "after_close must be True/False"
mismatch = earn[earn["announce_datetime"].dt.normalize() != earn["announce_date"]]
print("announce_date differs from announce_datetime date:", len(mismatch))

dups = earn.duplicated(["ticker", "announce_date"], keep=False)
print("Duplicate (ticker, announce_date) rows:", int(dups.sum()))
earn = earn.drop_duplicates(["ticker", "announce_date"], keep="last")

in_scope = earn["ticker"].isin(PROJECT_TICKERS) & earn["announce_date"].between(START, END)
print("Rows outside the project tickers / period (dropped):", int((~in_scope).sum()))
earn = earn[in_scope].copy()
print("Missing EPS values:", int(earn[["eps_estimate", "eps_actual", "surprise_pct"]].isna().sum().sum()))

## 3. Check `after_close`

Each company reports at a consistent time of day (e.g. AAPL, MSFT, NVDA after the close; JNJ, JPM, KO, XOM before the open). Events that break their company's usual pattern are listed so they can be reviewed.

Two events in the raw file carry a placeholder-looking timestamp (`AMZN 2024-08-01 12:00`, `TSLA 2021-10-20 15:00`) and are marked as before-close, but both companies reported **after** the close on those dates. They are corrected through the explicit `TIMING_OVERRIDES` list below. Add to it only after verifying a date against the company's earnings release.

In [ ]:
usual_after_close = earn.groupby("ticker")["after_close"].transform("mean") >= 0.5
outliers = earn[earn["after_close"] != usual_after_close]
print("Events that break their company's usual timing:")
display(outliers[["ticker", "announce_datetime", "after_close"]])

TIMING_OVERRIDES = {
    ("AMZN", "2024-08-01"): True,   # Q2 2024 results published after the close (raw time 12:00 is not the release time)
    ("TSLA", "2021-10-20"): True,   # Q3 2021 results published after the close (raw time 15:00 is not the release time)
}

applied = 0
for (ticker, day), value in TIMING_OVERRIDES.items():
    mask = (earn["ticker"] == ticker) & (earn["announce_date"] == pd.Timestamp(day))
    assert mask.sum() == 1, f"override {ticker} {day} matches {int(mask.sum())} rows"
    if earn.loc[mask, "after_close"].iloc[0] != value:
        earn.loc[mask, "after_close"] = value
        applied += 1
print("Timing overrides applied:", applied)

uncovered = outliers[~outliers.apply(lambda r: (r["ticker"], str(r["announce_date"].date())) in TIMING_OVERRIDES, axis=1)]
if len(uncovered):
    print("REVIEW: timing outliers without an override:")
    display(uncovered[["ticker", "announce_datetime", "after_close"]])

## 4. Reaction date for earnings

Recomputed from the announcement date and `after_close` with the trading calendar, then compared with the `reaction_date` supplied in the raw file. Only rows whose timing was corrected above should differ.

In [ ]:
earn["reaction_date_new"] = [reaction_date(d, a) for d, a in zip(earn["announce_date"], earn["after_close"])]
supplied = pd.to_datetime(earn["reaction_date"], errors="coerce")
changed = earn[earn["reaction_date_new"] != supplied]
print(f"Reaction dates that differ from the raw file: {len(changed)} of {len(earn)}")
display(changed[["ticker", "announce_date", "after_close", "reaction_date", "reaction_date_new"]])

# EPS sanity: surprise_pct comes from unrounded EPS, so only flag a clearly opposite sign.
gap = earn["eps_actual"] - earn["eps_estimate"]
opposite = earn[(gap.abs() >= 0.01) & (np.sign(gap) != np.sign(earn["surprise_pct"]))]
print("Rows where surprise_pct has the opposite sign of (actual - estimate):", len(opposite))
if len(opposite):
    display(opposite[["ticker", "announce_date", "eps_estimate", "eps_actual", "surprise_pct"]])

## 5. SEC 8-K filings

Keeps plain `8-K` forms (amendments such as `8-K/A` are reported and dropped), removes duplicate accession numbers, and cleans the item codes.

In [ ]:
k8 = pd.read_csv(K8_PATH, dtype=str)
print(f"Loaded {len(k8):,} 8-K rows")

required = {"ticker", "filing_date", "form", "items", "items_description", "accession_number"}
missing = required - set(k8.columns)
if missing:
    raise ValueError(f"events_8k.csv is missing columns: {sorted(missing)}")

k8["ticker"] = k8["ticker"].str.strip().str.upper()
k8["filing_date"] = pd.to_datetime(k8["filing_date"], errors="coerce").dt.normalize()
k8["accession_number"] = k8["accession_number"].str.strip()
assert k8["filing_date"].notna().all(), "unparseable filing_date"
assert k8["accession_number"].notna().all(), "missing accession_number"

print("Forms:", k8["form"].value_counts().to_dict())
k8 = k8[k8["form"].str.strip().str.upper() == "8-K"].copy()

print("Duplicate accession numbers dropped:", int(k8.duplicated("accession_number").sum()))
k8 = k8.drop_duplicates("accession_number", keep="last")

in_scope = k8["ticker"].isin(PROJECT_TICKERS) & k8["filing_date"].between(START, END)
print("Rows outside the project tickers / period (dropped):", int((~in_scope).sum()))
k8 = k8[in_scope].copy()

# Item codes: remove spaces ("2.02, 9.01" -> "2.02,9.01"); empty text becomes missing.
k8["items"] = k8["items"].str.replace(r"\s+", "", regex=True).replace("", pd.NA)
k8["items_description"] = k8["items_description"].str.strip().replace("", pd.NA)
print("Missing items / descriptions:", int(k8["items"].isna().sum()), "/", int(k8["items_description"].isna().sum()))

same_day = k8.duplicated(["ticker", "filing_date", "items"], keep=False).sum()
print("Rows sharing ticker + date + items with another filing (kept, different accession numbers):", int(same_day))

## 6. Reaction date for 8-K filings

Default: the filing date, or the next trading day if the filing date is not a trading day. 8-Ks with item `2.02` filed on the day of a known earnings announcement take that announcement's reaction date.

In [ ]:
k8["reaction_date"] = [next_trading_day(d) for d in k8["filing_date"]]

has_202 = k8["items"].fillna("").str.split(",").apply(lambda parts: "2.02" in parts)
earnings_ref = earn[["ticker", "announce_date", "reaction_date_new"]].rename(
    columns={"announce_date": "filing_date", "reaction_date_new": "earnings_reaction"}
)
k8 = k8.merge(earnings_ref, on=["ticker", "filing_date"], how="left")
align = has_202.to_numpy() & k8["earnings_reaction"].notna().to_numpy()
k8.loc[align, "reaction_date"] = k8.loc[align, "earnings_reaction"]
print(f"8-Ks with item 2.02: {int(has_202.sum())} | aligned to an earnings announcement: {int(align.sum())}")
k8 = k8.drop(columns="earnings_reaction")

## 7. Build the `events` table

In [ ]:
earnings_events = pd.DataFrame({
    "ticker": earn["ticker"],
    "event_type": "earnings",
    "event_date": earn["announce_date"],
    "reaction_date": earn["reaction_date_new"],
    "after_close": earn["after_close"].astype("boolean"),
    "eps_estimate": earn["eps_estimate"],
    "eps_actual": earn["eps_actual"],
    "surprise_pct": earn["surprise_pct"],
    "items": pd.NA,
    "items_description": pd.NA,
    "accession_number": pd.NA,
})

k8_events = pd.DataFrame({
    "ticker": k8["ticker"],
    "event_type": "8-K",
    "event_date": k8["filing_date"],
    "reaction_date": k8["reaction_date"],
    "after_close": pd.Series(pd.NA, index=k8.index, dtype="boolean"),
    "eps_estimate": np.nan,
    "eps_actual": np.nan,
    "surprise_pct": np.nan,
    "items": k8["items"],
    "items_description": k8["items_description"],
    "accession_number": k8["accession_number"],
})

events = pd.concat([earnings_events, k8_events], ignore_index=True)
events = events.sort_values(
    ["event_date", "ticker", "event_type", "accession_number"], na_position="first"
).reset_index(drop=True)
events.insert(0, "event_id", range(1, len(events) + 1))
print(f"{len(events):,} events: {events['event_type'].value_counts().to_dict()}")
display(events.head(8))

## 8. Validate

In [ ]:
assert events["event_id"].is_unique
assert events["ticker"].isin(PROJECT_TICKERS).all()
assert set(events["event_type"]) <= {"earnings", "8-K"}
assert not events[events["event_type"] == "earnings"].duplicated(["ticker", "event_date"]).any()
assert events.loc[events["event_type"] == "8-K", "accession_number"].is_unique

assert (events["reaction_date"] >= events["event_date"]).all(), "reaction_date before event_date"
assert events["reaction_date"].map(is_trading_day).all(), "reaction_date is not a trading day"
assert events["event_date"].between(START, END).all()

# Type-specific columns must be empty for the other type.
earnings_only = ["after_close", "eps_estimate", "eps_actual", "surprise_pct"]
k8_only = ["items", "items_description", "accession_number"]
assert events.loc[events["event_type"] == "8-K", earnings_only].isna().all().all()
assert events.loc[events["event_type"] == "earnings", k8_only].isna().all().all()

if COMPANIES_PATH.exists():
    known = set(pd.read_csv(COMPANIES_PATH, usecols=["ticker"], dtype=str)["ticker"])
    assert set(events["ticker"]) <= known, sorted(set(events["ticker"]) - known)
    print("All tickers exist in companies.csv.")
print("All validation checks passed.")

display(pd.crosstab(events["ticker"], events["event_type"], margins=True))
display(pd.crosstab(events["event_date"].dt.year, events["event_type"]))

## 9. Export

In [ ]:
export = events[[
    "event_id", "ticker", "event_type", "event_date", "reaction_date", "after_close",
    "eps_estimate", "eps_actual", "surprise_pct", "items", "items_description", "accession_number",
]].copy()
for col in ["event_date", "reaction_date"]:
    export[col] = export[col].dt.strftime("%Y-%m-%d")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
export.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(export):,} events to {OUTPUT_PATH.resolve()}")

## Completion checklist

- [ ] Row counts reviewed (earnings and 8-K per ticker and year).
- [ ] Timing outliers reviewed; every entry in `TIMING_OVERRIDES` verified against the earnings release.
- [ ] Reaction-date differences from the raw file are only the corrected rows.
- [ ] Calendar cross-check against `market_clean.csv` shows no unexplained differences.
- [ ] Team knows that each earnings event also has a matching 8-K with item `2.02` (avoid double counting).
- [ ] `events_clean.csv` shared with Person 4.